# AirPulse — Model Quality Monitoring & Retraining Pipeline

**MLOps Practical Assessment — 10 Marks**  
**Theme:** Extend your existing ML project with a model-quality monitoring and retraining capability.  
**Project:** [AirPulse: Hyperlocal Air Pollution Forecasting MLOps Project](https://github.com/SkandhHanda-Sk/AirPulse)  
**ML Task:** Regression — Continuous PM2.5 Concentration (µg/m³) 1-Hour Ahead Forecasting  
**Stations Covered:** `site_113` (Shadipur), `site_115` (NSIT Dwarka), `site_117` (ITO)  

---

## Practical Assessment Tasks Covered

| Task | Description | Status |
|---|---|---|
| **Task A** | **Prepare new data**: Load batch, validate 18 required features + target, check nulls/duplicates, display size & distribution | ✅ Implemented |
| **Task B** | **Evaluate existing model**: Load champion artifact, predict on new batch, calculate MAE/RMSE/R², save report | ✅ Implemented |
| **Task C** | **Retrain candidate model**: Merge historical train set + new batch, fit HistGradientBoosting pipeline, save candidate artifact, log MLflow run | ✅ Implemented |
| **Task D** | **Implement quality gate**: Evaluate candidate vs. champion on new batch, apply 2% MAE threshold, record decision in append-only JSONL, enforce human-in-the-loop | ✅ Implemented |
| **Task E** | **Demonstration and explanation**: End-to-end evidence summary, artifact verification table, architecture diagram & justification | ✅ Implemented |

## 0. Setup & Environment Imports

We configure relative paths from the repository root and import project modeling utilities.

In [ ]:
import os
import sys
import json
import math
import warnings
import datetime
from pathlib import Path

warnings.filterwarnings('ignore')
os.environ['GIT_PYTHON_REFRESH'] = 'quiet'

# Set repository root
REPO_ROOT = Path(os.path.abspath('')).parent if Path(os.path.abspath('')).name == 'monitoring' else Path(os.path.abspath(''))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from src.models.residual_hgb import (
    ANCHOR_COLUMN,
    CATEGORICAL_FEATURES,
    EXPECTED_FEATURES,
    HGB_PARAMS,
    NUMERIC_FEATURES,
    TARGET_COLUMN,
    build_residual_hgb_pipeline,
)
from monitoring.quality_gate import evaluate_gate, load_decision_log, TOLERANCE

# File Paths
CHAMPION_PATH       = REPO_ROOT / 'artifacts' / 'models' / 'residual_hgb_pipeline.joblib'
CANDIDATE_PATH      = REPO_ROOT / 'artifacts' / 'models' / 'candidate_model.joblib'
MODEL_INFO_PATH     = REPO_ROOT / 'artifacts' / 'models' / 'residual_hgb_model_info.json'
DATA_READY_DIR      = REPO_ROOT / 'data' / 'processed' / 'model_ready'
TRAIN_X_PATH        = DATA_READY_DIR / 'X_train.csv'
TRAIN_Y_PATH        = DATA_READY_DIR / 'y_train.csv'
VAL_X_PATH          = DATA_READY_DIR / 'X_validation.csv'
VAL_Y_PATH          = DATA_READY_DIR / 'y_validation.csv'

MONITORING_DIR      = REPO_ROOT / 'monitoring'
NEW_BATCH_PATH      = MONITORING_DIR / 'data' / 'new_batch.csv'
REPORTS_DIR         = MONITORING_DIR / 'reports'
EVAL_REPORT_PATH    = REPORTS_DIR / 'evaluation_report.json'
RETRAIN_REPORT_PATH = REPORTS_DIR / 'retrain_report.json'
DECISION_LOG_PATH   = REPORTS_DIR / 'quality_gate_decisions.jsonl'
MLRUNS_DIR          = REPO_ROOT / 'mlruns'

(MONITORING_DIR / 'data').mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

def compute_metrics(y_true, y_pred):
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(math.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))
    return {'mae': round(mae, 4), 'rmse': round(rmse, 4), 'r2': round(r2, 4)}

print('Setup complete. REPO_ROOT:', REPO_ROOT)
print('Champion Model Artifact Present:', CHAMPION_PATH.exists())
print('Historical Training Data Present:', TRAIN_X_PATH.exists())

---
## Task A — Prepare New Data

**Objective:**
1. Load incoming observation batch (real validation slice from XKDR Delhi PM2.5 network).
2. Inject validation test cases (anomalous nulls, duplicate records) to test the hygiene checks.
3. Validate all 18 input features + 1 target column (`target_pm25_next_hour`).
4. Check and handle missing values.
5. Check and deduplicate records.
6. Display cleaned dataset size and target distribution statistics (regression: mean, std, percentiles, station breakdown).

In [ ]:
# Load incoming batch
x_val = pd.read_csv(str(VAL_X_PATH))
y_val = pd.read_csv(str(VAL_Y_PATH))
raw_batch = x_val.copy()
raw_batch[TARGET_COLUMN] = y_val[TARGET_COLUMN]

# Inject sample anomaly records for audit testing (duplicates and nulls)
dup_rows = raw_batch.sample(4, random_state=42)
corrupted_batch = pd.concat([raw_batch, dup_rows], ignore_index=True)
dummy_null = raw_batch.iloc[0:2].copy()
dummy_null.loc[:, 'pm25_1h_before'] = np.nan
corrupted_batch = pd.concat([corrupted_batch, dummy_null], ignore_index=True)

print(f'Raw incoming batch before audit: {len(corrupted_batch):,} rows')

# 1. Column Validation
required_cols = EXPECTED_FEATURES + [TARGET_COLUMN]
missing_cols = [c for c in required_cols if c not in corrupted_batch.columns]
if missing_cols:
    raise ValueError(f'Schema Violation: Missing columns {missing_cols}')
print(f'✅ [Check 1/4] Schema Check: All {len(required_cols)} required columns present.')

# 2. Missing Value Validation
null_counts = corrupted_batch[required_cols].isnull().sum()
total_nulls = int(null_counts.sum())
print(f'🔍 [Check 2/4] Missing Values Check: Found {total_nulls} null entries.')
clean_batch = corrupted_batch.dropna(subset=required_cols).copy()
print(f'   Rows after removing null records: {len(clean_batch):,}')

# 3. Duplicate Validation
dups_count = int(clean_batch.duplicated().sum())
print(f'🔍 [Check 3/4] Duplicate Check: Found {dups_count} duplicate records.')
clean_batch = clean_batch.drop_duplicates().reset_index(drop=True)
print(f'   Rows after deduplication: {len(clean_batch):,}')

# 4. Dataset Size & Target Distribution
tgt = clean_batch[TARGET_COLUMN]
print(f'\n📊 [Check 4/4] Dataset Size & Target Distribution ({TARGET_COLUMN}):')
print(f'   Dataset Shape : {clean_batch.shape[0]:,} rows x {clean_batch.shape[1]} columns')
print(f'   Stations      : {sorted(clean_batch["station_id"].unique().tolist())}')
print(f'   Mean PM2.5    : {tgt.mean():.2f} µg/m³')
print(f'   Std Dev       : {tgt.std():.2f} µg/m³')
print(f'   Min PM2.5     : {tgt.min():.2f} µg/m³')
print(f'   25th %ile     : {np.percentile(tgt, 25):.2f} µg/m³')
print(f'   Median (50%)  : {tgt.median():.2f} µg/m³')
print(f'   75th %ile     : {np.percentile(tgt, 75):.2f} µg/m³')
print(f'   90th %ile     : {np.percentile(tgt, 90):.2f} µg/m³')
print(f'   Max PM2.5     : {tgt.max():.2f} µg/m³')

print('\nPer-Station Mean PM2.5 (Next Hour Target):')
for st, m in clean_batch.groupby('station_id')[TARGET_COLUMN].mean().items():
    print(f'   {st:<10}: {m:.2f} µg/m³')

# Save clean validated batch
clean_batch.to_csv(str(NEW_BATCH_PATH), index=False)
print(f'\n💾 Clean batch saved to: {NEW_BATCH_PATH}')

---
## Task B — Evaluate the Existing Model

**Objective:**
1. Load previously trained champion model (`artifacts/models/residual_hgb_pipeline.joblib`).
2. Generate next-hour predictions on the new validated batch.
3. Compute standard regression evaluation metrics: **MAE**, **RMSE**, and **R²**.
4. Compare performance against the Persistence Baseline (predict $t+1$ as current $t$ reading).
5. Save detailed evaluation results to `monitoring/reports/evaluation_report.json`.

In [ ]:
# 1. Load champion model
champion_model = joblib.load(str(CHAMPION_PATH))
print(f'✅ Champion model loaded: {CHAMPION_PATH.name} ({type(champion_model).__name__})')

# 2. Generate predictions on new batch
X_new = clean_batch[EXPECTED_FEATURES]
y_true = clean_batch[TARGET_COLUMN].to_numpy(dtype=float)
y_pred_champ = champion_model.predict(X_new)

# Persistence baseline
y_persist = clean_batch[ANCHOR_COLUMN].to_numpy(dtype=float)

# 3. Compute Metrics
champ_metrics = compute_metrics(y_true, y_pred_champ)
persist_metrics = compute_metrics(y_true, y_persist)

print('\nModel Evaluation Results on Incoming Batch:')
print('-' * 60)
print(f'{"Model":<35} {"MAE (µg/m³)":>12} {"RMSE (µg/m³)":>12} {"R²":>8}')
print('-' * 60)
print(f'{"Persistence Baseline":<35} {persist_metrics["mae"]:>12.4f} {persist_metrics["rmse"]:>12.4f} {persist_metrics["r2"]:>8.4f}')
print(f'{"Champion (Residual HGB)":<35} {champ_metrics["mae"]:>12.4f} {champ_metrics["rmse"]:>12.4f} {champ_metrics["r2"]:>8.4f}')
print('-' * 60)

# Per-station breakdown
per_station_eval = {}
print('\nPer-Station Champion Breakdown:')
for st in sorted(clean_batch['station_id'].unique()):
    mask = (clean_batch['station_id'] == st).to_numpy()
    sm = compute_metrics(y_true[mask], y_pred_champ[mask])
    per_station_eval[st] = sm
    print(f'   {st:<10}: MAE = {sm["mae"]:>7.4f} µg/m³ | RMSE = {sm["rmse"]:>7.4f} | R² = {sm["r2"]:>6.4f}')

# 4. Save evaluation report
eval_report = {
    'task': 'Task B -- Evaluate Existing Model',
    'timestamp_utc': datetime.datetime.utcnow().isoformat() + 'Z',
    'champion_artifact': str(CHAMPION_PATH),
    'evaluation_samples': int(len(y_true)),
    'metrics': champ_metrics,
    'persistence_baseline_metrics': persist_metrics,
    'station_breakdown': per_station_eval,
    'evaluation_dataset': str(NEW_BATCH_PATH),
}

with open(str(EVAL_REPORT_PATH), 'w', encoding='utf-8') as f:
    json.dump(eval_report, f, indent=2)

print(f'\n💾 Evaluation report saved to: {EVAL_REPORT_PATH}')

---
## Task C — Retrain the Model

**Objective:**
1. Combine existing training data (`data/processed/model_ready/X_train.csv` + `y_train.csv`: 15,016 samples) with the new labelled batch (3,629 samples) → total 18,645 training rows.
2. Train a new candidate model using the existing `build_residual_hgb_pipeline()`.
3. Save the candidate model separately (`artifacts/models/candidate_model.joblib`) without overwriting the champion.
4. Track the training run, parameters, and metrics in **MLflow** (`AirPulse_PM25_Forecasting` experiment).
5. Save detailed retraining report to `monitoring/reports/retrain_report.json`.

In [ ]:
# 1. Combine training data + new batch
X_train_orig = pd.read_csv(str(TRAIN_X_PATH))[EXPECTED_FEATURES]
y_train_orig = pd.read_csv(str(TRAIN_Y_PATH)).squeeze()

X_new_batch = clean_batch[EXPECTED_FEATURES]
y_new_batch = clean_batch[TARGET_COLUMN]

X_combined = pd.concat([X_train_orig, X_new_batch], ignore_index=True)
y_combined = pd.concat(
    [y_train_orig.reset_index(drop=True), y_new_batch.reset_index(drop=True)],
    ignore_index=True,
)
y_combined_arr = y_combined.to_numpy(dtype=float)

print(f'Historical Training Samples: {len(X_train_orig):,} rows')
print(f'New Batch Samples          : {len(X_new_batch):,} rows')
print(f'Combined Training Samples  : {len(X_combined):,} rows')

# 2. Build & train candidate model
print('\nFitting candidate HistGradientBoosting pipeline on combined data...')
candidate_model = build_residual_hgb_pipeline()
candidate_model.fit(X_combined, y_combined_arr)
print('✅ Candidate model training completed.')

# Evaluate candidate
y_cand_train_pred = candidate_model.predict(X_combined)
cand_train_metrics = compute_metrics(y_combined_arr, y_cand_train_pred)

y_cand_new_pred = candidate_model.predict(X_new_batch)
cand_new_metrics = compute_metrics(y_true, y_cand_new_pred)

print(f'Candidate Combined Train MAE: {cand_train_metrics["mae"]:.4f} µg/m³ (RMSE: {cand_train_metrics["rmse"]:.4f}, R²: {cand_train_metrics["r2"]:.4f})')
print(f'Candidate New Batch MAE     : {cand_new_metrics["mae"]:.4f} µg/m³ (RMSE: {cand_new_metrics["rmse"]:.4f}, R²: {cand_new_metrics["r2"]:.4f})')

# 3. Save candidate model artifact separately
joblib.dump(candidate_model, str(CANDIDATE_PATH))
print(f'\n💾 Candidate model saved separately to: {CANDIDATE_PATH}')

# 4. Track in MLflow
import mlflow
import mlflow.sklearn

mlruns_uri = MLRUNS_DIR.resolve().as_uri()
mlflow.set_tracking_uri(mlruns_uri)
mlflow.set_experiment('AirPulse_PM25_Forecasting')

with mlflow.start_run(run_name='retrain_candidate_pipeline') as run:
    cand_run_id = run.info.run_id
    mlflow.log_params(HGB_PARAMS)
    mlflow.log_param('total_training_rows', len(X_combined))
    mlflow.log_param('new_batch_rows', len(X_new_batch))
    mlflow.log_param('feature_count', len(EXPECTED_FEATURES))
    
    mlflow.log_metrics({
        'cand_train_mae': cand_train_metrics['mae'],
        'cand_train_rmse': cand_train_metrics['rmse'],
        'cand_train_r2': cand_train_metrics['r2'],
        'cand_new_batch_mae': cand_new_metrics['mae'],
        'cand_new_batch_rmse': cand_new_metrics['rmse'],
        'cand_new_batch_r2': cand_new_metrics['r2'],
    })
    
    try:
        mlflow.sklearn.log_model(candidate_model, artifact_path='candidate_model', serialization_format='pickle')
    except Exception:
        mlflow.log_artifact(str(CANDIDATE_PATH), artifact_path='model_artifact')
    mlflow.log_artifact(str(NEW_BATCH_PATH), artifact_path='data_batch')

print(f'🔬 MLflow Run Logged: {cand_run_id}')

# Save retrain report
retrain_report = {
    'task': 'Task C -- Retrain the Model',
    'timestamp_utc': datetime.datetime.utcnow().isoformat() + 'Z',
    'candidate_artifact': str(CANDIDATE_PATH),
    'historical_rows': len(X_train_orig),
    'new_batch_rows': len(X_new_batch),
    'total_training_rows': len(X_combined),
    'candidate_train_metrics': cand_train_metrics,
    'candidate_new_batch_metrics': cand_new_metrics,
    'mlflow_run_id': cand_run_id,
    'mlflow_experiment': 'AirPulse_PM25_Forecasting',
}
with open(str(RETRAIN_REPORT_PATH), 'w', encoding='utf-8') as f:
    json.dump(retrain_report, f, indent=2)
print(f'💾 Retraining report saved to: {RETRAIN_REPORT_PATH}')

---
## Task D — Implement a Quality Gate

**Objective:**
1. Compare existing champion model with newly trained candidate model on the incoming batch.
2. Implement decision rule:
   $$\text{candidate\_MAE} \le \text{champion\_MAE} \times 1.02$$
   - **If candidate meets criteria:** mark as `ELIGIBLE FOR PROMOTION`.
   - **Otherwise:** `RETAIN` existing champion.
   - **Guardrail:** Never automatically overwrite champion without human audit.
3. Log every decision into an append-only audit log: `monitoring/reports/quality_gate_decisions.jsonl`.

### Metric Selection Justification:
- **Why MAE (Mean Absolute Error)?**
  1. **Loss Function Alignment**: The estimator `HistGradientBoostingRegressor` is fitted using L1 absolute loss (`least_absolute_deviation` / `absolute_error`). Evaluating on MAE maintains rigorous internal consistency with the training objective.
  2. **Domain Interpretability**: MAE measures physical PM2.5 forecast error in $\mu\text{g/m}^3$, directly mapping onto WHO and Central Pollution Control Board (CPCB) AQI hazard bands.
  3. **Robustness to Delhi Air Pollution Spikes**: Delhi air pollution experiences extreme transient PM2.5 spikes (e.g. episodic biomass burning, diurnal inversions). RMSE squares these extreme residuals, penalizing models excessively for localized spikes. MAE provides a robust operational benchmark.
  4. **Seasonal Stability**: Unlike $R^2$ which fluctuates depending on seasonal PM2.5 variance (monsoon vs winter inversions), MAE provides a stable, year-round metric.

In [ ]:
# Head-to-Head Comparison
print(f'{"Metric":<10} {"Champion":>12} {"Candidate":>12} {"Delta":>10} {"Superior Model":>15}')
print('-' * 65)
for m in ['mae', 'rmse', 'r2']:
    c_val = champ_metrics[m]
    k_val = cand_new_metrics[m]
    d_val = k_val - c_val
    sign = '+' if d_val > 0 else ''
    winner = 'Candidate ✅' if (k_val < c_val if m != 'r2' else k_val > c_val) else 'Champion 🛡️'
    print(f'{m.upper():<10} {c_val:>12.4f} {k_val:>12.4f} {sign}{d_val:>9.4f} {winner:>15}')

# Evaluate Gate
gate_result = evaluate_gate(
    champion_metrics=champ_metrics,
    candidate_metrics=cand_new_metrics,
    champion_artifact=str(CHAMPION_PATH),
    candidate_artifact=str(CANDIDATE_PATH),
    tolerance=TOLERANCE,
)

decision = gate_result['decision']
print('\n' + '=' * 60)
print(f'QUALITY GATE DECISION: [ {decision} ]')
print(f'Reason: {gate_result["reason"]}')
print('=' * 60)

if decision == 'PROMOTE':
    print('\n🚀 Candidate is ELIGIBLE FOR PROMOTION.')
    print('   Safety Guardrail: Champion artifact is NOT automatically replaced.')
    print('   In production CI/CD, this prompts human approval to update MLflow alias:')
    print('   client.set_registered_model_alias("airpulse-pm25-forecast", "champion", new_version)')
else:
    print('\n🛡️ Champion RETAINED. No production changes made.')

print(f'\n📜 Decision appended to audit log: {DECISION_LOG_PATH}')

---
## Task E — Demonstration and Explanation

**Objective:**
Demonstrate:
1. Successful execution of the workflow.
2. Evaluation report generation.
3. Candidate model artifact creation.
4. Quality gate decision audit.
5. Complete architectural explanation of the workflow.

In [ ]:
print('=' * 75)
print('TASK E -- WORKFLOW DEMONSTRATION & ARTIFACT VERIFICATION')
print('=' * 75)

artifacts = [
    ('Task A: Validated Batch Data', NEW_BATCH_PATH),
    ('Task B: Champion Evaluation Report', EVAL_REPORT_PATH),
    ('Task C: Retraining Report', RETRAIN_REPORT_PATH),
    ('Task C: Candidate Model Artifact', CANDIDATE_PATH),
    ('Task D: Quality Gate Decision Log', DECISION_LOG_PATH),
    ('Reference: Champion Model Artifact', CHAMPION_PATH),
]

print(f'{"Artifact Description":<35} {"File Path":<45} {"Status":>12}')
print('-' * 95)
for label, path in artifacts:
    exists = path.exists()
    size = f'{path.stat().st_size:,} bytes' if exists else 'NOT FOUND'
    status = f'PASSED ({size})' if exists else 'FAILED'
    rel = str(path.relative_to(REPO_ROOT))
    print(f'{label:<35} {rel:<45} {status:>12}')

print('\n' + '=' * 75)
print('DECISION AUDIT TRAIL (Last Entries in JSONL):')
print('-' * 75)
for entry in load_decision_log()[-3:]:
    print(f'[{entry["timestamp_utc"]}] Decision: {entry["decision"]} | Cand MAE: {entry["candidate_mae"]} vs Champ MAE: {entry["champion_mae"]}')
print('=' * 75)
print('✅ ALL FIVE TASKS (A, B, C, D, E) EXECUTED AND DEMONSTRATED SUCCESSFULLY.')

## Complete Workflow Architecture

```
                          AIRPULSE CONTINUOUS MLOps LIFECYCLE
                                                                    
  [ Production Traffic ]                                            
            │                                                       
            ▼                                                       
     Hourly Readings (CPCB / XKDR Stations: Shadipur, Dwarka, ITO)  
            │                                                       
            ▼                                                       
  ┌───────────────────────┐                                         
  │ TASK A: Batch Intake  │  • Validate 18 lag/rolling features + target
  │ & Data Validation     │  • Null imputation & duplicate rejection
  └──────────┬────────────┘  • Output: monitoring/data/new_batch.csv
             │                                                      
             ▼                                                      
  ┌───────────────────────┐                                         
  │ TASK B: Model Quality │  • Load artifacts/models/residual_hgb_pipeline.joblib
  │ Evaluation (Champion) │  • Compute MAE, RMSE, R² against persistence
  └──────────┬────────────┘  • Output: monitoring/reports/evaluation_report.json
             │                                                      
             ▼                                                      
  ┌───────────────────────┐                                         
  │ TASK C: Candidate     │  • Combine historical (15,016) + new (3,629) = 18,645 rows
  │ Retraining Pipeline   │  • Fit HistGradientBoostingRegressor via existing pipeline
  └──────────┬────────────┘  • Track params, metrics & artifacts in MLflow
             │               • Output: artifacts/models/candidate_model.joblib
             ▼                                                      
  ┌───────────────────────┐                                         
  │ TASK D: Quality Gate  │  • Decision Rule: Cand MAE ≤ Champ MAE × 1.02
  │ Safety Governance     │  • Never overwrite champion automatically
  └──────────┬────────────┘  • Append to monitoring/reports/quality_gate_decisions.jsonl
             │                                                      
       ┌─────┴───────────────────────┐                              
       │                             │                              
  [ PROMOTE ]                   [ RETAIN ]                          
       │                             │                              
       ▼                             ▼                              
  Human Approval Gate           Champion Preserved                  
  Update MLflow 'champion'      Candidate Archived for Analysis     
  Redeploy FastAPI Service                                          
```